###### Feature Engineering

In [1]:
import pandas as pd
import numpy as np

# Load the raw data (Assuming you have downloaded these from Kaggle)
app_train = pd.read_csv('../data/raw/application_train.csv')
bureau = pd.read_csv('../data/raw/bureau.csv')


In [2]:
# View

app_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
app_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 122 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(41), object(16)
memory usage: 286.2+ MB


In [4]:
bureau.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [5]:
bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_CURR              int64  
 1   SK_ID_BUREAU            int64  
 2   CREDIT_ACTIVE           object 
 3   CREDIT_CURRENCY         object 
 4   DAYS_CREDIT             int64  
 5   CREDIT_DAY_OVERDUE      int64  
 6   DAYS_CREDIT_ENDDATE     float64
 7   DAYS_ENDDATE_FACT       float64
 8   AMT_CREDIT_MAX_OVERDUE  float64
 9   CNT_CREDIT_PROLONG      int64  
 10  AMT_CREDIT_SUM          float64
 11  AMT_CREDIT_SUM_DEBT     float64
 12  AMT_CREDIT_SUM_LIMIT    float64
 13  AMT_CREDIT_SUM_OVERDUE  float64
 14  CREDIT_TYPE             object 
 15  DAYS_CREDIT_UPDATE      int64  
 16  AMT_ANNUITY             float64
dtypes: float64(8), int64(6), object(3)
memory usage: 222.6+ MB


In [6]:
# Feature Engineering: Bureau Aggregations

bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'DAYS_CREDIT': ['min', 'max', 'mean'],
    'CREDIT_DAY_OVERDUE': ['max', 'mean'],
    'AMT_CREDIT_SUM': ['sum', 'mean'],
    'CNT_CREDIT_PROLONG': ['sum']
})

In [9]:
# verifying
bureau_agg.head()

DAYS_CREDIT                   CREDIT_DAY_OVERDUE       \
                   min  max         mean                max mean   
SK_ID_CURR                                                         
100001           -1572  -49  -735.000000                  0  0.0   
100002           -1437 -103  -874.000000                  0  0.0   
100003           -2586 -606 -1400.750000                  0  0.0   
100004           -1326 -408  -867.000000                  0  0.0   
100005            -373  -62  -190.666667                  0  0.0   

           AMT_CREDIT_SUM                CNT_CREDIT_PROLONG  
                      sum           mean                sum  
SK_ID_CURR                                                   
100001        1453365.000  207623.571429                  0  
100002         865055.565  108131.945625                  0  
100003        1017400.500  254350.125000                  0  
100004         189037.800   94518.900000                  0  
100005         657126.000  219042.000000                  0

In [10]:
# Flatten the MultiIndex columns
bureau_agg.columns = ['BU_'.join(col).strip() for col in bureau_agg.columns.values]


In [12]:
# verifying
bureau_agg.head()

,DAYS_CREDITBU_min,DAYS_CREDITBU_max,DAYS_CREDITBU_mean,CREDIT_DAY_OVERDUEBU_max,CREDIT_DAY_OVERDUEBU_mean,AMT_CREDIT_SUMBU_sum,AMT_CREDIT_SUMBU_mean,CNT_CREDIT_PROLONGBU_sum
SK_ID_CURR,,,,,,,,
100001,-1572,-49,-735.000000,0,0.0,1453365.000,207623.571429,0
100002,-1437,-103,-874.000000,0,0.0,865055.565,108131.945625,0
100003,-2586,-606,-1400.750000,0,0.0,1017400.500,254350.125000,0
100004,-1326,-408,-867.000000,0,0.0,189037.800,94518.900000,0
100005,-373,-62,-190.666667,0,0.0,657126.000,219042.000000,0


In [13]:
# Merge with Main Application
train_merged = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')

# Handle Missing Values (Common in thin-file borrowers)
# For thin-file users, no bureau data means 0 debt, not 'NaN'
train_merged.fillna(0, inplace=True)

print(f"Final shape of the engineered dataset: {train_merged.shape}")
train_merged.head()

Final shape of the engineered dataset: (307511, 130)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,DAYS_CREDITBU_min,DAYS_CREDITBU_max,DAYS_CREDITBU_mean,CREDIT_DAY_OVERDUEBU_max,CREDIT_DAY_OVERDUEBU_mean,AMT_CREDIT_SUMBU_sum,AMT_CREDIT_SUMBU_mean,CNT_CREDIT_PROLONGBU_sum
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0.0,1.0,-1437.0,-103.0,-874.00,0.0,0.0,865055.565,108131.945625,0.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0.0,0.0,-2586.0,-606.0,-1400.75,0.0,0.0,1017400.500,254350.125000,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0.0,0.0,-1326.0,-408.0,-867.00,0.0,0.0,189037.800,94518.900000,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.000,0.000000,0.0
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0.0,0.0,-1149.0,-1149.0,-1149.00,0.0,0.0,146250.000,146250.000000,0.0


In [14]:
train_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 130 entries, SK_ID_CURR to CNT_CREDIT_PROLONGBU_sum
dtypes: float64(73), int64(41), object(16)
memory usage: 305.0+ MB


###### Creating a one year column

In [18]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_processor import DataProcessor

In [19]:
# Assuming 'today' is 2026 for the sake of this project
CURRENT_YEAR = 2026

processor = DataProcessor()
macro_df = processor.fetch_world_bank_data()

# Transforming macro_df to get the 2024/2025 values
# (Since YR2025 might be NaN, take YR2024)
latest_inflation = macro_df.loc[macro_df['series'] == 'FP.CPI.TOTL.ZG', 'YR2024'].values[0]
latest_gdp = macro_df.loc[macro_df['series'] == 'NY.GDP.MKTP.KD.ZG', 'YR2024'].values[0]

train_merged['macro_inflation'] = latest_inflation
train_merged['macro_gdp'] = latest_gdp

print(f"Added Macro Features: Inflation ({latest_inflation:.2f}%), GDP ({latest_gdp:.2f}%)")

Fetching data for NGA...
Added Macro Features: Inflation (33.24%), GDP (4.06%)


###### Encoding 

In [20]:
# Identifying Categorical Columns
cat_cols = train_merged.select_dtypes(include=['object']).columns
print(f"Encoding {len(cat_cols)} categorical columns...")


Encoding 16 categorical columns...


In [21]:
# Factorizing (Label Encoding) 
# This converts ['Cash loans', 'Revolving loans'] -> [0, 1]
for col in cat_cols:
    train_merged[col], _ = pd.factorize(train_merged[col])


In [22]:
# Handling any remaining NaNs or Infs (LightGBM is robust, but clean is better)
train_merged.replace([np.inf, -np.inf], np.nan, inplace=True)
train_merged.fillna(0, inplace=True)

In [23]:
# verifying
train_merged.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,DAYS_CREDITBU_min,DAYS_CREDITBU_max,DAYS_CREDITBU_mean,CREDIT_DAY_OVERDUEBU_max,CREDIT_DAY_OVERDUEBU_mean,AMT_CREDIT_SUMBU_sum,AMT_CREDIT_SUMBU_mean,CNT_CREDIT_PROLONGBU_sum,macro_inflation,macro_gdp
0,100002,1,0,0,0,0,0,202500.0,406597.5,24700.5,...,-1437.0,-103.0,-874.00,0.0,0.0,865055.565,108131.945625,0.0,33.242097,4.062364
1,100003,0,0,1,0,1,0,270000.0,1293502.5,35698.5,...,-2586.0,-606.0,-1400.75,0.0,0.0,1017400.500,254350.125000,0.0,33.242097,4.062364
2,100004,0,1,0,1,0,0,67500.0,135000.0,6750.0,...,-1326.0,-408.0,-867.00,0.0,0.0,189037.800,94518.900000,0.0,33.242097,4.062364
3,100006,0,0,1,0,0,0,135000.0,312682.5,29686.5,...,0.0,0.0,0.00,0.0,0.0,0.000,0.000000,0.0,33.242097,4.062364
4,100007,0,0,0,0,0,0,121500.0,513000.0,21865.5,...,-1149.0,-1149.0,-1149.00,0.0,0.0,146250.000,146250.000000,0.0,33.242097,4.062364


In [24]:
# Save the "Golden Dataset"
output_path = '../data/processed/train_final.csv'
train_merged.to_csv(output_path, index=False)

print(f"Final shape: {train_merged.shape}")
print(f"Data saved to: {output_path}")

Final shape: (307511, 132)
Data saved to: ../data/processed/train_final.csv
